# Phase 1: Ingestion & Environment

In [1]:
import pandas as pd     
import sqlite3

In [2]:
import os
print(os.getcwd())

c:\Users\rites\OneDrive\Desktop\expense-audit-flow\notebooks


In [3]:
df = pd.read_csv('../data/fraudTrain.csv')
print("Dataset loaded successfully")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Dataset loaded successfully
Rows: 1296675, Columns: 23


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [4]:
print(df['is_fraud'].value_counts())
print("\nFraud percentage:")
print(df['is_fraud'].value_counts(normalize=True) * 100)

is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Fraud percentage:
is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64


In [5]:
df = df.rename(columns={'is_fraud': 'flagged'})
print(df['flagged'].value_counts())

flagged
0    1289169
1       7506
Name: count, dtype: int64


In [6]:
import sqlite3
conn = sqlite3.connect('../data/expense_audit.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)
print("Data loaded into SQLite successfully")

Data loaded into SQLite successfully


# Phase 2: Discovery & SQL Auditing

In [ ]:
query = "SELECT * FROM transactions LIMIT 5"
result = pd.read_sql_query(query, conn)
result

In [9]:
query = """
SELECT flagged, COUNT(*) as total_count
FROM transactions
GROUP BY flagged
"""
result = pd.read_sql_query(query, conn)
result

,flagged,total_count
0,0,1289169
1,1,7506


In [10]:
query = """
SELECT merchant, category, amt, city, state
FROM transactions
WHERE flagged = 1
LIMIT 10
"""
result = pd.read_sql_query(query, conn)
result

,merchant,category,amt,city,state
0,fraud_Rutherford-Mertz,grocery_pos,281.06,Collettsville,NC
1,"fraud_Jenkins, Hauck and Friesen",gas_transport,11.52,San Antonio,TX
2,fraud_Goodwin-Nitzsche,grocery_pos,276.31,San Antonio,TX
3,fraud_Erdman-Kertzmann,gas_transport,7.03,Collettsville,NC
4,fraud_Koepp-Parker,grocery_pos,275.73,San Antonio,TX
5,fraud_Medhurst PLC,shopping_net,844.80,Collettsville,NC
6,fraud_Ruecker Group,misc_net,843.91,Collettsville,NC
7,fraud_Conroy-Cruickshank,gas_transport,10.76,San Antonio,TX
8,fraud_Koepp-Parker,grocery_pos,332.35,San Antonio,TX
9,fraud_Strosin-Cruickshank,grocery_pos,315.34,San Antonio,TX


Merchant Risk Analysis

In [11]:
query = """
SELECT merchant, COUNT(*) as incident_count
FROM transactions
WHERE flagged = 1
GROUP BY merchant
ORDER BY incident_count DESC
LIMIT 10
"""
result = pd.read_sql_query(query, conn)
result

,merchant,incident_count
0,fraud_Rau and Sons,49
1,fraud_Kozey-Boehm,48
2,fraud_Cormier LLC,48
3,fraud_Vandervort-Funk,47
4,fraud_Kilback LLC,47
5,fraud_Doyle Ltd,47
6,fraud_Padberg-Welch,44
7,fraud_Kuhn LLC,44
8,fraud_Terry-Huel,43
9,fraud_Koepp-Witting,42


Category Risk Analysis

In [12]:
query = """
SELECT category, COUNT(*) as incident_count
FROM transactions
WHERE flagged = 1
GROUP BY category
ORDER BY incident_count DESC
"""
result = pd.read_sql_query(query, conn)
result

,category,incident_count
0,grocery_pos,1743
1,shopping_net,1713
2,misc_net,915
3,shopping_pos,843
4,gas_transport,618
5,misc_pos,250
6,kids_pets,239
7,entertainment,233
8,personal_care,220
9,home,198


High value flagged transactions

In [13]:
query = """
SELECT merchant, category, amt, city, state, flagged
FROM transactions
WHERE amt > 1000
ORDER BY amt DESC
LIMIT 10
"""
result = pd.read_sql_query(query, conn)
result

,merchant,category,amt,city,state,flagged
0,fraud_Satterfield-Lowe,travel,28948.90,Westerville,NE,0
1,"fraud_Monahan, Hermann and Johns",travel,27390.12,Armonk,NY,0
2,"fraud_Monahan, Hermann and Johns",travel,27119.77,Amorita,OK,0
3,fraud_Boyer-Haley,travel,26544.12,Cross,SC,0
4,fraud_Hackett Group,travel,25086.94,Lawrence,MA,0
5,fraud_Tillman LLC,travel,17897.24,Clarks Mills,PA,0
6,"fraud_Reichel, Bradtke and Blanda",travel,15305.95,Lakeland,FL,0
7,fraud_Goyette-Herzog,travel,15047.03,San Antonio,TX,0
8,"fraud_Larson, Quitzon and Spencer",travel,15034.18,Clayton,OK,0
9,fraud_Lynch-Mohr,travel,14849.74,Afton,MN,0
